# CS60075 Project 10: Open-domain QA

Runs the real TriviaQA experiment end to end. Upload the submission ZIP and the separate data snapshot together in the first picker. CPU is supported; models and Wikipedia require internet. The optional corpus snapshot restores the exact measured collection. All stages run as fresh Python subprocesses to avoid stale notebook imports.

**Default:** pinned CPU environment, 300 unique validation questions, seed 42. No model training.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, subprocess, sys, os, json

uploaded = files.upload()
for name in uploaded:
    if name.endswith(".zip"):
        with zipfile.ZipFile(name) as z:
            # Reject archive entries outside the current working directory.
            base = Path.cwd().resolve()
            for entry in z.infolist():
                if not (base / entry.filename).resolve().is_relative_to(base):
                    raise ValueError("Unsafe ZIP entry")
            z.extractall(base)
project = Path("/content/nlp_qa_project")
assert (project / "config.json").exists(), "Upload NLP_QA_Submission.zip"
os.chdir(project)
os.environ["PYTHONHASHSEED"] = "42"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("Project:", project)

## Install the pinned CPU environment

The reference run uses CPU. Keep this default to reproduce it. You may use an optional GPU wheel below, but its outputs are a separate device run. Installation may take several minutes.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
# OPTIONAL GPU: run the following only if a CUDA runtime is selected.
# subprocess.run([sys.executable, "-m", "pip", "install", "torch==2.6.0", "--index-url", "https://download.pytorch.org/whl/cu124", "--force-reinstall", "--no-deps"], check=True)
# The CUDA run was not validated as part of the reference CPU experiment.
subprocess.run([sys.executable, "-m", "pytest", "tests", "-q"], check=True)

## Optional: restore the exact corpus snapshot

The default restores the exact corpus. If you uploaded NLP_QA_Data_Snapshot.zip with the submission ZIP in the first picker, no second upload is needed. Set RESTORE_SNAPSHOT=False for a fresh live Wikipedia collection. Live pages can change, so a live rebuild is a new experiment.

In [ ]:
RESTORE_SNAPSHOT = True
if RESTORE_SNAPSHOT and not (project / "data/corpus.jsonl").exists():
    snapshots = files.upload()
    for name in snapshots:
        if name.endswith(".zip"):
            with zipfile.ZipFile(name) as z:
                base = project.parent.resolve()
                for entry in z.infolist():
                    if not (base / entry.filename).resolve().is_relative_to(base):
                        raise ValueError("Unsafe ZIP entry")
                z.extractall(base)

# A live collection has new evidence. Preserve reference audit labels separately.
if not RESTORE_SNAPSHOT:
    labels = project / "results/support_labels.csv"
    if labels.exists():
        from datetime import datetime
        archived = labels.with_name("reference_support_labels_" + datetime.now().strftime("%Y%m%d_%H%M%S") + ".csv")
        labels.rename(archived)
        print("Reference support labels archived. Review the new support audit before reporting its accuracy.")

def run(stage):
    subprocess.run([sys.executable, "-u", "code/run.py", stage], check=True)

# DATA PREPARATION: fixed question sample, then one shared frozen corpus.
run("prepare")

## Task 1 - BM25 and extraction

Runs independently after preparation. It retrieves 50 candidates for evaluation, uses the top passage for answering, and saves every output.

In [ ]:
run("task1")

## Task 2 - Dense retrieval and RAG

Frozen MiniLM, normalized FAISS flat search, top 5 evidence passages, 200 model tokens per passage, greedy generation, max 64 new tokens.

In [ ]:
run("task2")

## Task 3 - Hybrid retrieval and support warnings

RRF constant 60, candidate depth 50, same answer generator. Analysis also applies the overlap check to original Task 2 outputs.

In [ ]:
run("task3")

## Tables, plots, qualitative examples, and PDF report

After changing the corpus, model, prompt, or predictions, independently re-review support labels. Remove stale support_labels.csv for a changed experiment; do not reuse old support judgments.

In [ ]:
run("analyze")
run("report")
from IPython.display import display, Markdown, Image
display(Markdown(Path("results/comparison.md").read_text()))
display(Image(filename="results/comparison.png"))

## Download the new run

This archives your current code, results, report, and data manifests while excluding large datasets and checkpoints. The separate snapshot is needed for exact data restoration.

In [ ]:
subprocess.run([sys.executable, "code/package_submission.py"], check=True)
files.download("../NLP_QA_Submission.zip")
files.download("report.pdf")